In [0]:
storage_account_name = "cbdalakehouse1"
storage_account_key = "actual_key"
container_name = "data"

# Configure access
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

## Define base path

In [0]:
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

## Implementation

In [0]:
dbutils.library.restartPython()

In [0]:
# Import required libraries
from pyspark.sql.functions import col, substring, regexp_replace, when, expr, sum as _sum, year, to_date, corr, avg, count, lit, trim, upper
from pyspark.sql.types import IntegerType, DoubleType
import mlflow
import pandas as pd
from sklearn.linear_model import LinearRegression

# ── UPDATED CREDENTIALS ──
storage_account_name = "cbdalakehouse1"
storage_account_key = "actual_key"
container_name = "data"

# Configure Spark to access Azure Data Lake Storage
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

# Read source data
df_crime     = spark.read.csv(base_path + "bronze/crime/MPS Borough Level Crime (Historical).csv",     header=True, inferSchema=True)
df_fly       = spark.read.csv(base_path + "bronze/flytipping/fly-tipping-borough.csv",    header=True, inferSchema=True)
df_recycling = spark.read.csv(base_path + "bronze/recycling/household-recycling-borough.csv", header=True, inferSchema=True)

# ── CRIME: Unpivot wide → long ──
value_columns = [c for c in df_crime.columns if c not in ["BoroughName", "MajorText", "MinorText"]]
stack_expr_crime = f"stack({len(value_columns)}, " + ", ".join([f"'{c}', `{c}`" for c in value_columns]) + ") as (date, crime_count)"

df_crime_long = df_crime.select("BoroughName", "MajorText", "MinorText", expr(stack_expr_crime))

df_crime_clean = df_crime_long \
    .withColumn("date",        to_date(col("date").cast("string"), "yyyyMM")) \
    .withColumn("year",        year(col("date"))) \
    .withColumn("crime_count", when(col("crime_count").isNull(), lit(0)).otherwise(col("crime_count").cast("int"))) \
    .filter(col("BoroughName").isNotNull()) \
    .filter(col("date").isNotNull()) \
    .dropna(subset=["BoroughName", "crime_count"])

# ── FLY-TIPPING: Fix year format "2011-12" → 2011, remove commas from numbers ──
df_fly_clean = df_fly \
    .withColumn("year", when(col("year").isNotNull(),
        substring(col("year").cast("string"), 1, 4).cast("int")).otherwise(lit(None))) \
    .withColumn("total_incidents", when(col("total_incidents").rlike("^[0-9,]+$"),
        regexp_replace(col("total_incidents").cast("string"), ",", "").cast("int")).otherwise(lit(0))) \
    .filter(col("area").isNotNull()) \
    .withColumnRenamed("area", "BoroughName")

df_fly_clean = df_fly_clean.withColumn("BoroughName", regexp_replace(col("BoroughName"), " COUNCIL", ""))
median_incidents = df_fly_clean.selectExpr("percentile_approx(total_incidents, 0.5)").collect()[0][0]
df_fly_clean = df_fly_clean.fillna({"total_incidents": median_incidents})

# ── RECYCLING: Rename columns to match pipeline
df_recycling_clean = df_recycling \
    .withColumnRenamed("Area",           "BoroughName") \
    .withColumnRenamed("Recycling_Rates","recycling_rate") \
    .withColumnRenamed("Year",           "year_raw") \
    .withColumn("year",           substring(col("year_raw"), 1, 4).cast(IntegerType())) \
    .withColumn("recycling_rate", col("recycling_rate").cast(DoubleType())) \
    .drop("year_raw", "Code")

# ── Standardise borough names to UPPERCASE for joining ──
df_crime_clean     = df_crime_clean.withColumn(    "BoroughName", upper(trim(col("BoroughName"))))
df_fly_clean       = df_fly_clean.withColumn(      "BoroughName", upper(trim(col("BoroughName"))))
df_recycling_clean = df_recycling_clean.withColumn("BoroughName", upper(trim(col("BoroughName"))))

# ── Aggregate to borough-year level ──
df_crime_yearly     = df_crime_clean.groupBy(    "BoroughName", "year").agg(_sum("crime_count").alias("total_crimes"))
df_fly_yearly       = df_fly_clean.groupBy(      "BoroughName", "year").agg(_sum("total_incidents").alias("total_fly_tipping"))
df_recycling_yearly = df_recycling_clean.groupBy("BoroughName", "year").agg(avg("recycling_rate").alias("recycling_rate"))

# ── Inner join all three datasets ──
df_joined = df_crime_yearly \
    .join(df_fly_yearly,       on=["BoroughName", "year"], how="inner") \
    .join(df_recycling_yearly, on=["BoroughName", "year"], how="inner")

print(f"Joined rows:  {df_joined.count()}")
print(f"Boroughs:     {df_joined.select('BoroughName').distinct().count()}")
print("Sample of joined data:")
df_joined.show(10)

# ── Correlations ──
correlation_crime_fly       = df_joined.select(corr("total_crimes", "total_fly_tipping")).collect()[0][0]
correlation_crime_recycling = df_joined.select(corr("total_crimes", "recycling_rate")).collect()[0][0]
correlation_fly_recycling   = df_joined.select(corr("total_fly_tipping", "recycling_rate")).collect()[0][0]

print(f"\nCorrelation (Crime vs Fly-tipping):    {correlation_crime_fly:.4f}")
print(f"Correlation (Crime vs Recycling Rate): {correlation_crime_recycling:.4f}")
print(f"Correlation (Fly-tipping vs Recycling):{correlation_fly_recycling:.4f}")

# ── Regression with MLflow ──
with mlflow.start_run(run_name="Broken_Windows_Analysis_Three_Datasets"):
    df_pd = df_joined.toPandas()

    X1 = df_pd[["total_fly_tipping"]].fillna(0)
    y  = df_pd["total_crimes"].fillna(0)
    model1 = LinearRegression().fit(X1, y)

    X2 = df_pd[["recycling_rate"]].fillna(0)
    model2 = LinearRegression().fit(X2, y)

    X3 = df_pd[["total_fly_tipping", "recycling_rate"]].fillna(0)
    model3 = LinearRegression().fit(X3, y)

    mlflow.log_metric("correlation_crime_fly",       correlation_crime_fly)
    mlflow.log_metric("correlation_crime_recycling",  correlation_crime_recycling)
    mlflow.log_metric("correlation_fly_recycling",    correlation_fly_recycling)
    mlflow.log_metric("r2_crime_fly",                 model1.score(X1, y))
    mlflow.log_metric("r2_crime_recycling",           model2.score(X2, y))
    mlflow.log_metric("r2_multiple_regression",       model3.score(X3, y))
    mlflow.log_param("coefficient_fly",       model1.coef_[0])
    mlflow.log_param("coefficient_recycling", model2.coef_[0])
    mlflow.log_param("intercept",             model1.intercept_)

    print(f"\nR² (Crime vs Fly-tipping only):   {model1.score(X1, y):.4f}")
    print(f"R² (Crime vs Recycling only):     {model2.score(X2, y):.4f}")
    print(f"R² (Crime vs BOTH — final model): {model3.score(X3, y):.4f}")
    print(f"\nRegression equation:")
    print(f"Crime = {model3.coef_[0]:.4f} * Fly_tipping + {model3.coef_[1]:.4f} * Recycling_Rate + {model3.intercept_:.2f}")

# ── Write Gold layer ──
df_joined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(base_path + "gold/broken_windows_analysis")
print(f"\n✓ Gold layer written to: {base_path}gold/broken_windows_analysis")


In [0]:
# ── SILVER LAYER ──
df_crime_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "silver/crime")

df_fly_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "silver/fly_tipping")

df_recycling_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "silver/recycling")

print("✓ Silver layer written")
print(f"  silver/crime:      {df_crime_clean.count()} rows")
print(f"  silver/fly_tipping:{df_fly_clean.count()} rows")
print(f"  silver/recycling:  {df_recycling_clean.count()} rows")

In [0]:
from pyspark.sql.functions import col, stddev, mean, round as _round, min as _min, max as _max, corr, when
from sklearn.linear_model import LinearRegression
import mlflow

print("BROKEN WINDOWS THEORY ANALYSIS - RESULTS")

crime_min = df_joined.agg(_min("total_crimes")).collect()[0][0]
crime_max = df_joined.agg(_max("total_crimes")).collect()[0][0]
fly_min = df_joined.agg(_min("total_fly_tipping")).collect()[0][0]
fly_max = df_joined.agg(_max("total_fly_tipping")).collect()[0][0]
recycle_min = df_joined.agg(_min("recycling_rate")).collect()[0][0]
recycle_max = df_joined.agg(_max("recycling_rate")).collect()[0][0]

print(f"Total crime range: {crime_min:.0f} - {crime_max:.0f}")
print(f"Fly-tipping range: {fly_min:.0f} - {fly_max:.0f}")
print(f"Recycling rate range: {recycle_min:.1f} - {recycle_max:.1f}")

print("\nTOP 5 BOROUGHS WITH HIGHEST CRIME:")
df_joined.orderBy(col("total_crimes").desc()).select("BoroughName", "year", "total_crimes").show(5)

print("\nTOP 5 BOROUGHS WITH HIGHEST FLY-TIPPING:")
df_joined.orderBy(col("total_fly_tipping").desc()).select("BoroughName", "year", "total_fly_tipping", "recycling_rate").show(5)

print("\nTOP 5 BOROUGHS WITH HIGHEST RECYCLING RATES:")
df_joined.orderBy(col("recycling_rate").desc()).select("BoroughName", "year", "recycling_rate", "total_crimes").show(5)

print(f"Crime vs Fly-tipping correlation: {correlation_crime_fly:.4f}")
print(f"Crime vs Recycling Rate correlation: {correlation_crime_recycling:.4f}")
print(f"Fly-tipping vs Recycling Rate correlation: {correlation_fly_recycling:.4f}")

with mlflow.start_run(run_name="Broken_Windows_Final_Analysis"):
    df_pd = df_joined.toPandas()
    
    X = df_pd[["total_fly_tipping", "recycling_rate"]].fillna(0)
    y = df_pd["total_crimes"].fillna(0)
    model = LinearRegression()
    model.fit(X, y)
    
    print(f"R² Score: {model.score(X, y):.4f}")
    print(f"Regression: Crime = {model.coef_[0]:.2f} * Fly_tipping + {model.coef_[1]:.2f} * Recycling_Rate + {model.intercept_:.2f}")
    
    mlflow.log_metric("final_r2", model.score(X, y))
    mlflow.log_param("crime_fly_coefficient", model.coef_[0])
    mlflow.log_param("crime_recycling_coefficient", model.coef_[1])
    mlflow.log_param("intercept", model.intercept_)

df_trend = df_joined.groupBy("year").agg(
    corr("total_crimes", "total_fly_tipping").alias("crime_fly_corr"),
    corr("total_crimes", "recycling_rate").alias("crime_recycling_corr")
).orderBy("year")

print("\nYearly correlations:")
df_trend.show(10)

df_risk = df_joined \
    .withColumn("crime_risk", 
        when(col("total_crimes") > 40000, "VERY HIGH")
        .when(col("total_crimes") > 30000, "HIGH")
        .when(col("total_crimes") > 20000, "MEDIUM")
        .otherwise("LOW")) \
    .withColumn("neglect_score",
        when(col("total_fly_tipping") > 20000, "HIGH NEGLECT")
        .when(col("total_fly_tipping") > 10000, "MEDIUM NEGLECT")
        .otherwise("LOW NEGLECT"))

df_risk.groupBy("crime_risk", "neglect_score").count().show()

df_risk.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(base_path + "gold/broken_windows_with_risk")

print("Analysis complete")

In [0]:
from scipy import stats
import statsmodels.api as sm

df_pd = df_joined.toPandas()

# Correlation p-values
r_fly, p_fly = stats.pearsonr(df_pd["total_crimes"], df_pd["total_fly_tipping"])
r_rec, p_rec = stats.pearsonr(df_pd["total_crimes"], df_pd["recycling_rate"])

print(f"Crime vs Fly-tipping: r = {r_fly:.4f}, p = {p_fly:.2e}")
print(f"Crime vs Recycling:   r = {r_rec:.4f}, p = {p_rec:.2e}")

# Regression p-values
X = sm.add_constant(df_pd[["total_fly_tipping", "recycling_rate"]].fillna(0))
y = df_pd["total_crimes"].fillna(0)
model = sm.OLS(y, X).fit()

print(f"\nR² = {model.rsquared:.4f}")
print(f"Adjusted R² = {model.rsquared_adj:.4f}")
print(f"F = {model.fvalue:.2f}, p = {model.f_pvalue:.2e}")
print(f"n = {len(df_pd)}")
print(f"\nCoefficients:")
for name in model.params.index:
    print(f"  {name}: coef={model.params[name]:.4f}, p={model.pvalues[name]:.2e}")

In [0]:
# ── EXPORT CSV FOR TABLEAU ──
df_risk.select(
    "BoroughName", "year", "total_crimes",
    "total_fly_tipping", "recycling_rate",
    "crime_risk", "neglect_score"
) \
.withColumnRenamed("total_fly_tipping", "total_incidents") \
.coalesce(1) \
.write.mode("overwrite") \
.option("header", "true") \
.csv(base_path + "gold/tableau_export")

print("✓ Tableau CSV exported to gold/tableau_export/")